# UBELIX - Working with .MAT Files

Complete guide and useful commands for processing .MAT files on UBELIX cluster

## Opening MAT Files (Load MAT)

To open and load .mat files in Python, you have several options depending on the file version and size. Here's a comparison table:

| Aspect | scipy.io.loadmat | mat73.loadmat | h5py |
|--------|---|---|---|
| **Version** | < 7.3 (older or small files) | > 7.3 (newer or very large files) | Any HDF5 .mat file |
| **Internal Format** | Classic binary | HDF5 | HDF5 |
| **Dependencies** | scipy | mat73, hdf5 | h5py |
| **Memory Management** | Loads entire file into memory | Optimized for large files | Low-level, fine-grained control |
| **Use Case** | Simple loading | General use | Detailed data exploration & attribute access |

### Three Loading Approaches:

1. **scipy.io.loadmat**: Best for small, pre-v7.3 .mat files. Simple to use but loads entire file into memory.
2. **mat73.loadmat**: Best for modern v7.3 HDF5-based .mat files. Higher-level interface optimized for large files.
3. **h5py**: Low-level HDF5 access for detailed exploration of file structure and specific attributes (as shown below).

### Smart Loading Strategy (used by mat_to_pkl_ji_brain.py)

The script uses a `load_ji_mat()` function that intelligently tries both formats:
- First attempts `scipy.io.loadmat` (fast for legacy files)
- If it fails, falls back to `mat73.loadmat` (for v7.3 HDF5 files)
- Returns both the loaded data dictionary AND the loader type used

This approach handles both file formats transparently without needing to know the file version in advance.

**Important:** The script [mat_to_pkl_ji_brain.py](mat_to_pkl_ji_brain.py) in this workspace handles both .mat file formats automatically using this smart loading approach, so it's good to understand all three methods for troubleshooting and data manipulation.

## Complete Workflow: Setting Up Environment and Launching Jobs

### Step 1: Create and Activate a Conda Environment

First, create a new conda environment with all required packages:

```bash
# Create a new environment with Python and key packages
conda create -n myenv python=3.9 scipy h5py

# Install additional packages if needed
conda activate myenv
pip install mat73
```

Once created, the environment `myenv` will be available on UBELIX and can be activated in your SLURM scripts with `source activate myenv`.

### Step 2: Create Your SLURM Script (.sh file)

Create a shell script with the SLURM directives and your Python command:

```bash
# Save this as: submit_job.sh
#!/bin/bash
#SBATCH --cpus-per-task=1
#SBATCH --mem-per-cpu=300G
#SBATCH --time=0-04:00:00
#SBATCH --output=bash/outputs/output_%j.out
#SBATCH --error=bash/outputs/error_%j.err

# Activate your conda environment
source activate myenv

# Run your Python script
python -u your_script.py
```

### Step 3: Submit Your Job with sbatch

Once your `.sh` file is ready and uploaded to UBELIX, submit the job:

```bash
# First, navigate to your working directory on UBELIX
cd /storage/homefs/ab25c720/MicroBrain/XiangJi

# Submit the job
sbatch submit_job.sh

# You'll see output like:
# Submitted batch job 2671256

# Capture the Job ID for tracking
```

### Step 4: Monitor Your Job

Track your job's progress using the commands from section 5:

```bash
# Check job status
sacct -j 2671256 --format=JobID,State,ExitCode,Elapsed,MaxRSS

# View output while running or after completion
tail -n 50 bash/outputs/output_2671256.out
tail -n 50 bash/outputs/error_2671256.err

# Or view the complete output
cat bash/outputs/output_2671256.out
```

### Complete Workflow Summary

1. **Create environment** → `conda create -n myenv python=3.9 scipy h5py mat73`
2. **Activate locally** → `conda activate myenv` (for testing)
3. **Create .sh script** → Write SLURM directives + your Python commands
4. **Upload to UBELIX** → `scp your_script.sh user@submit03.unibe.ch:/path/`
5. **Submit job** → `sbatch your_script.sh`
6. **Monitor progress** → `sacct -j JOBID` and check output files

## 2. Running SLURM Scripts (.sh)

Once you have your environment set up and your Python script ready, create a SLURM submission script to run your jobs on the cluster.

### Example SLURM Script for Whole Brain Processing

```bash
#!/bin/bash
#SBATCH --cpus-per-task=1
#SBATCH --mem-per-cpu=200G
#SBATCH --time=0-03:00:00
#SBATCH --output=bash/outputs/output_%j.out
#SBATCH --error=bash/outputs/error_%j.err

source ~/myenv/bin/activate
python -u mat_to_pkl_whole_brain.py
```

**To submit the job:**
```bash
sbatch run_wholebrain.sh
```

**Important Notes:**
- Adjust `--cpus-per-task`, `--mem-per-cpu`, and `--time` based on your job requirements
- If the job fails, try increasing memory or time limits
- The `%j` in output/error paths will be replaced with the job ID

For more info about the resources, check the [UBELIX User Guide](https://hpc-unibe-ch.github.io/runjobs/scheduled-jobs/slurm-quickstart/)

## 3. Checking Job Status

After submitting your job, you can monitor its progress using these commands:

### Basic Job Status Check
```bash
squeue -j <JOBID>
```

### Detailed Job Information
```bash
sacct -j <JOBID> --format=JobID,JobName,State,ExitCode,MaxRSS,Elapsed
```

**Example Output:**
```
JobID           JobName      State ExitCode     MaxRSS    Elapsed 
------------ ---------- ---------- -------- ---------- ---------- 
1588183      run_whole+    RUNNING      0:0              00:59:32 
1588183.bat+      batch    RUNNING      0:0              00:59:32 
1588183.ext+     extern    RUNNING      0:0              00:59:32
```

**Common States:**
- `PENDING`: Waiting in queue
- `RUNNING`: Currently executing
- `COMPLETED`: Finished successfully
- `FAILED`: Terminated with error

### Example Output: Submitting and Checking a Job

When you submit a job using `sbatch`, you'll see output like this:

```
[ab25c720@submit03 XiangJi]$ sbatch scaling_check.sh
sbatch: Partition set to "epyc2"
sbatch: QoS set to "job_gratis"
sbatch: --------------------------------------------------
sbatch: This job generates no costs!
sbatch: --------------------------------------------------
Submitted batch job 2671256

[ab25c720@submit03 XiangJi]$ sacct -j 2671256 --format=JobID,State,ExitCode,Elapsed,MaxRSS
JobID                 State ExitCode   Elapsed     MaxRSS
---------- --------------- --------- ---------- ----------
2671256              PENDING      0:0   00:00:00
```

**Key information:**
- **Job ID**: 2671256 (use this to track your job)
- **Partition**: epyc2 (assigned automatically)
- **State**: PENDING (waiting to start), can also be RUNNING, COMPLETED, or FAILED
- **ExitCode**: 0 (success), non-zero indicates an error

## 4. Checking Job Logs (Output and Errors)

Monitor your job's progress by checking the output and error log files:

### View Recent Output
```bash
tail -n 50 bash/outputs/output_<JOBID>.out
tail -n 50 bash/outputs/error_<JOBID>.err
```

### View Complete Logs
```bash
cat bash/outputs/output_<JOBID>.out
cat bash/outputs/error_<JOBID>.err
```

**Important:** Look for these error indicators:
- `Killed`: Job was terminated (usually due to time or memory limits)
- `OOM`: Out of Memory error
- `Traceback`: Python error with stack trace
- Non-zero `ExitCode`: Indicates failure

### Complete Monitoring Script
```bash
JOBID=1717848
sacct -j $JOBID --format=JobID,JobName,State,ExitCode,MaxRSS,Elapsed
tail -n 30 bash/outputs/output_${JOBID}.out
tail -n 30 bash/outputs/error_${JOBID}.err
```

### Example: Job Failure Due to TypeError

**Job Status:**
```
JobID           JobName      State ExitCode     MaxRSS    Elapsed 
------------ ---------- ---------- -------- ---------- ---------- 
1717848      pkl2vtp_w+     FAILED      1:0              01:26:20 
1717848.bat+      batch     FAILED      1:0 148236152K   01:26:20 
1717848.ext+     extern  COMPLETED      0:0       122K   01:26:21
```

**Error Log:**
```
Traceback (most recent call last):
  File "/storage/homefs/ab25c720/MicroBrain/XiangJi/pkl2vtp_mvn.py", line 321, in <module>
    write_vtp(graph, output_path+'WholeBrain_ML_2018_08_15_whole_brain_graph_straight.vtp', tortuous=False) # NON TORTUOUS
  File "/storage/homefs/ab25c720/MicroBrain/XiangJi/pkl2vtp_mvn.py", line 301, in write_vtp
    WriteOnFileVTP(
  File "/storage/homefs/ab25c720/MicroBrain/XiangJi/pkl2vtp_mvn.py", line 53, in WriteOnFileVTP
    data.text = " ".join(["{:10.15e}".format(r) for r in data_array])
  File "/storage/homefs/ab25c720/MicroBrain/XiangJi/pkl2vtp_mvn.py", line 53, in <listcomp>
    data.text = " ".join(["{:10.15e}".format(r) for r in data_array])
TypeError: unsupported format string passed to NoneType.__format__
```

**Note:** Output file naming can vary (`output_%j.out`, `vtp_%j.out`, `pkl_%j.out`). Use consistent naming in your SLURM scripts for easier monitoring.

## 5. Activating Conda Environment

Always activate your conda environment before running Python scripts:

```bash
source ~/myenv/bin/activate
```

**Important:**
- Your environment must contain all required packages (scipy, h5py, mat73, etc.)
- Activation is required in both interactive sessions and SLURM scripts
- Use `conda list` to verify installed packages

## 6. Loading Data to UBELIX

Transfer files from your local machine to UBELIX using `scp` (run from your local terminal, not UBELIX):

### Upload Single File
```bash
scp ~/folder/file.py ab25c720@submit03.unibe.ch:~/project/
```

### Upload Entire Folder
```bash
scp -r ~/folder ab25c720@submit03.unibe.ch:~/project/
```

### Upload to Specific Directory
```bash
scp ~/folder/file.py ab25c720@submit03.unibe.ch:/storage/homefs/ab25c720/MicroBrain/XiangJi/
```

### For Large Datasets (use rsync)
```bash
rsync -avz ~/folder/ ab25c720@submit03.unibe.ch:~/project/
```

**Verification:**
- After upload, SSH to UBELIX and use `ls` to confirm files arrived
- Your data storage location: `/storage/homefs/ab25c720/MicroBrain/XiangJi/`

**Note:** All file paths in your scripts must point to UBELIX storage locations, not local paths.

### Important Notes on Script Execution in UBELIX

It is important to activate the conda environment, which must include all required packages for your script. 

    - The Python script MUST be available in UBELIX (either pre-installed or available through modules). 

    - All file paths accessed by your script MUST point to UBELIX storage locations, NOT local paths.

    - All dependencies (packages) MUST be installed in the 'myenv' environment before running the job.

Once your job completes, you can examine the `.out` and `.err` files to see the output or any errors that occurred during execution.

## JI.MAT INFO


## 1. KEYS 

In [ ]:
import h5py

path = "/storage/homefs/ab25c720/MicroBrain/XiangJi/WholeBrain_ML_2018_08_15_whole_brain_graph.mat"

with h5py.File(path, "r") as f:
    print(list(f.keys()))

## 2. Know Keys Inside STAT_DATA

To see all keys inside the `stat_data` group:

In [ ]:
import h5py

path = "/storage/homefs/ab25c720/MicroBrain/XiangJi/WholeBrain_ML_2018_08_15_whole_brain_graph.mat"

with h5py.File(path, "r") as f:
    print(list(f["stat_data"].keys()))

## 3. Know the Brain Volume in mm³

To retrieve the brain volume value:

In [ ]:
import h5py

path = "/storage/homefs/ab25c720/MicroBrain/XiangJi/WholeBrain_ML_2018_08_15_whole_brain_graph.mat"

with h5py.File(path, "r") as f:
    x = f["stat_data"]["brain_volume_mm3"]
    print(float(x[0,0]))